# What is Machine Learning — Exercises

Companion to the note [What is Machine Learning](What%20is%20Machine%20Learning.md).

Practise Mitchell's E/T/P definition, the regularized empirical-risk objective $\hat\theta = \arg\min_\theta \frac1N\sum_i \ell(f_\theta(x_i), y_i) + \lambda\Omega(\theta)$, common loss functions, the ingredients of a learning algorithm and the main task types, by hand and in NumPy.

**17 exercises** · 🧠 Concept ×5 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.datasets import load_iris, load_digits
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, log_loss, hinge_loss
from sklearn.model_selection import train_test_split
from scipy import stats
rng = np.random.default_rng(0)

## Part A · Concepts

### Exercise 1 · Experience, Task, Performance
*🧠 Concept · ★☆☆*

For each system, name **E** (experience), **T** (task) and **P** (performance measure) in Mitchell's definition.

1. An e-mail spam filter that learns from messages users mark as spam.
2. A chess program that improves by playing games against itself.
3. A model predicting house prices from size, location and age.
4. A recommender that orders products on a shop's front page.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

T is *what the program does*, P is *a number that says how well*, E is *the data it learns from*.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| System | E | T | P |
|---|---|---|---|
| Spam filter | e-mails labelled spam / not spam by users | classify new e-mails | accuracy, or better precision/recall on spam |
| Chess | games of self-play (states, moves, outcomes) | choose moves | win rate (or Elo) against opponents |
| House prices | past sales with features and sale price | regression: predict price | RMSE / MAE on unseen houses |
| Recommender | logged clicks / purchases | rank products | click-through rate, NDCG, revenue |

Note how P must be measured on **new** data: improving on the training data alone is not learning (see [[Overfitting and Regularization]]).

</details>

### Exercise 2 · Rules or learning?
*🧠 Concept · ★☆☆*

"Instead of hand-writing rules, we fit a function to data." For which of these problems is ML a good idea, and for which
is a hand-written program better? Justify briefly.

(a) Computing VAT on an invoice. (b) Recognising handwritten digits. (c) Detecting credit-card fraud whose patterns change monthly.
(d) Sorting a list of numbers. (e) Predicting which customers will cancel their subscription.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

ML pays off when the rule is unknown or too complex to write down, when it changes over time, and when labelled data exists.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- (a), (d): **rules**. The exact algorithm is known, must be 100 % correct, and needs no data.
- (b): **ML**. Nobody can write down the rule "what makes a 7", but labelled examples are easy to collect.
- (c): **ML**. Patterns drift, so a model that is retrained on fresh data adapts; hand rules go stale.
- (e): **ML**. The relationship between behaviour and churn is unknown and statistical; historical data provides labels.

</details>

### Exercise 3 · Name the task type
*🧠 Concept · ★☆☆*

Classify each problem as classification, regression, clustering, density estimation, dimensionality reduction or
sequential decision making.

1. Predict tomorrow's temperature in °C. 2. Group news articles by topic without labels. 3. Decide whether a tumour is malignant.
4. Compress 784-pixel images into 2 numbers for plotting. 5. Flag transactions with very low probability under a fitted model.
6. Control a robot arm to pick up objects.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Is there a target? Is it discrete or continuous? Does the agent act and receive rewards over time?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1 regression (continuous target) · 2 clustering · 3 classification · 4 dimensionality reduction (e.g. [[PCA]]) ·
5 density estimation (anomaly = low $p(x)$) · 6 sequential decision making ([[RL Basics and MDPs]]).

</details>

### Exercise 4 · The four ingredients
*🧠 Concept · ★★☆*

Fill in the note's table (representation, objective, optimization) for: (a) [[Linear Regression]], (b) [[Logistic Regression]],
(c) [[Decision Trees]], (d) [[k-Nearest Neighbors]], (e) a Gaussian mixture model. Which of them does **not** really fit the
"minimise a loss" template, and why?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

k-NN has no training step at all. A tree is built greedily, not by global optimisation.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| Model | Representation | Objective | Optimization |
|---|---|---|---|
| Linear regression | $w^\top x + b$ | MSE | closed form (normal equations) or GD |
| Logistic regression | $\sigma(w^\top x + b)$ | cross-entropy (log-likelihood) | GD / Newton (no closed form) |
| Decision tree | axis-aligned partition | impurity (Gini / entropy) | greedy splits |
| k-NN | the stored training set | none at training time | none (lazy); work happens at prediction |
| GMM | mixture of Gaussians | log-likelihood | EM |

k-NN is **non-parametric and lazy**: it does not fit $\theta$ by minimising a loss; the only choice ($k$, distance) is a hyperparameter
picked by validation. Trees minimise a loss only greedily and locally.

</details>

### Exercise 5 · Empirical vs. expected risk
*🧠 Concept · ★★☆*

The note minimises the **empirical** risk $\hat R(\theta)=\frac1N\sum_i \ell(f_\theta(x_i),y_i)$. What we actually care about is the
**expected** risk $R(\theta)=\mathbb E_{(x,y)\sim p}[\ell(f_\theta(x),y)]$.

(a) Why can we not minimise $R$ directly? (b) Why is $\hat R(\hat\theta)$ an optimistic estimate of $R(\hat\theta)$?
(c) Name two things in the objective or workflow that help close the gap.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\hat\theta$ was chosen *because* it makes $\hat R$ small on those very samples.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $p(x,y)$ is unknown; we only have samples, so we replace the expectation by an average (law of large numbers).
(b) For a fixed $\theta$, $\hat R$ is unbiased. But $\hat\theta$ is selected to minimise $\hat R$ on the same data, so it also fits the noise
of that sample: $\mathbb E[\hat R(\hat\theta)] \le R(\hat\theta)$ (the **generalization gap**).
(c) The regularizer $\lambda\Omega(\theta)$ (smaller effective hypothesis class), more data, and estimating $R$ on held-out data
([[Cross-Validation and Model Selection]]) rather than on the training set.

</details>

## Part B · By hand

### Exercise 6 · Empirical risk of a given model
*✍️ By hand · ★☆☆*

Model $f(x) = 2x + 1$, data $x = (0,1,2,3)$, $y = (1.5, 2.5, 5.5, 6.0)$.
Compute the empirical risk with (a) squared loss (MSE) and (b) absolute loss (MAE).

In [ ]:
mse_hand = None
mae_hand = None

_x = np.array([0,1,2,3.]); _y = np.array([1.5,2.5,5.5,6.0])
check('MSE', mse_hand, mean_squared_error(_y, 2*_x+1))
check('MAE', mae_hand, mean_absolute_error(_y, 2*_x+1))

<details>
<summary><b>💡 Hint</b></summary>

Predictions are $(1,3,5,7)$; residuals $y - f(x) = (0.5,-0.5,0.5,-1)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Residuals $(0.5,-0.5,0.5,-1)$. (a) MSE $= (0.25+0.25+0.25+1)/4 = 1.75/4 = 0.4375$. (b) MAE $= (0.5+0.5+0.5+1)/4 = 0.625$.

```python
mse_hand = 0.4375
mae_hand = 0.625
```

</details>

### Exercise 7 · Adding the regularizer
*✍️ By hand · ★☆☆*

Same model, viewed as $\theta = (b, w) = (1, 2)$. Use squared loss, $\Omega(\theta) = w^2$ (the bias is not penalised) and
$\lambda = 0.1$. Compute the full objective $\frac1N\sum_i\ell + \lambda\Omega(\theta)$.
Would the model $\theta'=(1.4, 1.6)$ have a larger or smaller penalty term?

In [ ]:
objective_hand = None

check('regularized objective', objective_hand, mean_squared_error([1.5,2.5,5.5,6.0], 2*np.arange(4)+1) + 0.1*2**2)

<details>
<summary><b>💡 Hint</b></summary>

Reuse the MSE from the previous exercise.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$0.4375 + 0.1\cdot 2^2 = 0.4375 + 0.4 = 0.8375$. For $\theta'$ the penalty is $0.1\cdot1.6^2 = 0.256$, smaller: the regularizer prefers
smaller slopes, trading some data fit for simplicity.

```python
objective_hand = 0.8375
```

</details>

### Exercise 8 · Best constant predictor
*✍️ By hand · ★★☆*

Fit the simplest hypothesis class, a constant $f(x) = c$, to $y = (1, 2, 2, 3, 12)$.

(a) Show that minimising $\sum_i (y_i - c)^2$ gives $c = \bar y$. (b) Argue that minimising $\sum_i |y_i - c|$ gives the median.
(c) Compute both and store them. Which is more affected by the outlier 12?

In [ ]:
c_squared = None
c_absolute = None

_yy = np.array([1,2,2,3,12.])
from scipy.optimize import minimize_scalar
check('squared loss', c_squared, minimize_scalar(lambda c: np.sum((_yy-c)**2)).x, tol=1e-4)
check('absolute loss', c_absolute, minimize_scalar(lambda c: np.sum(np.abs(_yy-c)), bounds=(0, 12), method='bounded').x, tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Differentiate $\sum (y_i-c)^2$ w.r.t. $c$ and set to zero.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

The derivative of $\sum|y_i - c|$ is $\#\{y_i < c\} - \#\{y_i > c\}$: it is zero where half the points are on each side.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\frac{d}{dc}\sum(y_i-c)^2 = -2\sum(y_i - c) = 0 \Rightarrow c = \frac1N\sum y_i$.
(b) The subgradient is (#points below $c$) − (#points above $c$); it changes sign at the median.
(c) Mean $= 20/5 = 4$, median $= 2$. The mean is pulled toward the outlier; absolute loss is robust. This is why the
choice of $\ell$ is part of the model.

```python
c_squared = 4.0
c_absolute = 2.0
```

</details>

### Exercise 9 · Cross-entropy by hand
*✍️ By hand · ★☆☆*

A binary classifier outputs $\hat p = P(y=1\mid x)$. For labels $y = (1, 0, 1)$ and predictions $\hat p = (0.9, 0.2, 0.6)$ compute
the average cross-entropy $-\frac1N\sum [y\ln\hat p + (1-y)\ln(1-\hat p)]$ (natural log). Which sample contributes most?

In [ ]:
ce_hand = None

check('cross-entropy', ce_hand, log_loss([1,0,1], [0.9,0.2,0.6]), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Each term is $-\ln$ of the probability assigned to the **true** class: $0.9$, $0.8$, $0.6$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$-\frac13(\ln 0.9 + \ln 0.8 + \ln 0.6) = \frac13(0.1054 + 0.2231 + 0.5108) = 0.2798$.
The third sample (only 60 % on the true class) contributes most. A confidently wrong prediction ($\hat p\to0$ for $y=1$) would give $\to\infty$.

```python
ce_hand = -(np.log(0.9) + np.log(0.8) + np.log(0.6)) / 3
```

</details>

### Exercise 10 · Hinge loss vs. 0-1 loss
*✍️ By hand · ★☆☆*

With labels $y\in\{-1,+1\}$ and real-valued scores $f(x)$, the hinge loss is $\max(0, 1 - y f(x))$.
For $y = (+1, -1, +1, -1)$ and $f = (2, -0.5, 0.3, 0.4)$ compute the mean hinge loss and the mean 0-1 loss (error rate of $\operatorname{sign} f$).
Why does hinge loss penalise sample 3 even though it is classified correctly?

In [ ]:
hinge_hand = None
zero_one_hand = None

_yh = np.array([1,-1,1,-1]); _f = np.array([2,-0.5,0.3,0.4])
check('hinge', hinge_hand, hinge_loss(_yh, _f))
check('0-1', zero_one_hand, np.mean(np.sign(_f) != _yh))

<details>
<summary><b>💡 Hint</b></summary>

Compute the margins $y_i f(x_i) = (2, 0.5, 0.3, -0.4)$ first.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Margins $(2, 0.5, 0.3, -0.4)$, hinge terms $(0, 0.5, 0.7, 1.4)$, mean $= 2.6/4 = 0.65$. Only sample 4 has a negative margin,
so the 0-1 loss is $0.25$. Hinge loss demands a margin of at least 1 ([[Support Vector Machines]]): sample 3 is right but not
*confidently* right. Hinge is a convex upper bound on 0-1 loss, which makes it optimisable.

```python
hinge_hand = 0.65
zero_one_hand = 0.25
```

</details>

### Exercise 11 · Minimising a regularized objective in closed form
*✍️ By hand · ★★☆*

For a 1-D model without intercept, $f_w(x) = wx$, minimise $J(w) = \frac1N\sum_i (w x_i - y_i)^2 + \lambda w^2$.
(a) Derive $\hat w$ in closed form. (b) Evaluate it for $x=(1,2,3)$, $y=(2,3,7)$, $\lambda=0.5$. (c) What happens as $\lambda\to\infty$ and $\lambda\to0$?

In [ ]:
w_hat = None

_x3 = np.array([1,2,3.]); _y3 = np.array([2,3,7.])
check('w_hat', w_hat, Ridge(alpha=3*0.5, fit_intercept=False).fit(_x3[:,None], _y3).coef_[0], tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$J'(w) = \frac2N\sum_i x_i(wx_i - y_i) + 2\lambda w$. Multiply through by $N/2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $J'(w)=0 \Rightarrow w\sum x_i^2 - \sum x_iy_i + N\lambda w = 0 \Rightarrow \hat w = \dfrac{\sum x_iy_i}{\sum x_i^2 + N\lambda}$.
(b) $\sum x_iy_i = 2+6+21 = 29$, $\sum x_i^2 = 14$, $N\lambda = 1.5$: $\hat w = 29/15.5 \approx 1.871$.
(c) $\lambda\to\infty$: $\hat w\to0$ (all bias). $\lambda\to0$: least squares, $29/14\approx2.071$.
Note sklearn's `Ridge(alpha)` uses the **unnormalised** sum, so `alpha` $= N\lambda$.

```python
w_hat = 29 / 15.5
```

</details>

## Part C · Code

### Exercise 12 · A library of loss functions
*💻 Code · ★☆☆*

Implement the four losses as functions returning the **mean** over samples:
`squared(y, yhat)`, `absolute(y, yhat)`, `cross_entropy(y, p)` (labels 0/1, clip $p$ to $[10^{-12}, 1-10^{-12}]$),
`hinge(y, score)` (labels ±1). They are checked against sklearn on random data.

In [ ]:
def squared(y, yhat):
    return None
def absolute(y, yhat):
    return None
def cross_entropy(y, p):
    return None
def hinge(y, score):
    return None

_yr = rng.normal(size=50); _yp = rng.normal(size=50)
_yb = rng.integers(0, 2, 50); _pb = rng.uniform(0.01, 0.99, 50); _s = rng.normal(size=50)
check('squared', squared(_yr, _yp), mean_squared_error(_yr, _yp))
check('absolute', absolute(_yr, _yp), mean_absolute_error(_yr, _yp))
check('cross-entropy', cross_entropy(_yb, _pb), log_loss(_yb, _pb))
check('hinge', hinge(2*_yb-1, _s), hinge_loss(2*_yb-1, _s))

<details>
<summary><b>💡 Hint</b></summary>

All are one-liners with `np.mean`. Use `np.maximum(0, 1 - y*score)` for the hinge.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Clipping avoids `log(0) = -inf` when a model outputs exactly 0 or 1.

```python
def squared(y, yhat):
    return np.mean((y - yhat) ** 2)
def absolute(y, yhat):
    return np.mean(np.abs(y - yhat))
def cross_entropy(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
def hinge(y, score):
    return np.mean(np.maximum(0, 1 - y * score))
```

</details>

### Exercise 13 · Regularized ERM with gradient descent
*💻 Code · ★★☆*

Minimise $J(w)=\frac1N\lVert Xw-y\rVert^2 + \lambda\lVert w\rVert^2$ (no intercept) with gradient descent:
implement `erm_gd(X, y, lam, lr, epochs)`. Check it against `Ridge(alpha=N*lam, fit_intercept=False)`
(recall the scaling from the closed-form exercise).

In [ ]:
def erm_gd(X, y, lam=0.1, lr=0.1, epochs=2000):
    # TODO: gradient = (2/N) X^T (Xw - y) + 2 lam w
    return None

X_e = rng.normal(size=(80, 4))
y_e = X_e @ np.array([1.0, -2.0, 0.0, 3.0]) + 0.3 * rng.normal(size=80)
w_erm = erm_gd(X_e, y_e, lam=0.1)

check('GD = ridge', w_erm, Ridge(alpha=80*0.1, fit_intercept=False).fit(X_e, y_e).coef_, tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Start at `w = np.zeros(d)`, update `w -= lr * grad` for `epochs` steps.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\nabla J = \frac2N X^\top(Xw-y) + 2\lambda w$. Setting it to zero gives $(X^\top X + N\lambda I)w = X^\top y$, which is exactly ridge with
`alpha` $=N\lambda$. This is the general recipe of the note: choose $f_\theta$, $\ell$, $\Omega$, then optimise
([[Gradient Descent]]).

```python
def erm_gd(X, y, lam=0.1, lr=0.1, epochs=2000):
    N, d = X.shape
    w = np.zeros(d)
    for _ in range(epochs):
        w -= lr * (2 / N * X.T @ (X @ w - y) + 2 * lam * w)
    return w

X_e = rng.normal(size=(80, 4))
y_e = X_e @ np.array([1.0, -2.0, 0.0, 3.0]) + 0.3 * rng.normal(size=80)
w_erm = erm_gd(X_e, y_e, lam=0.1)
```

</details>

### Exercise 14 · Hand-written rules vs. a learned model
*💻 Code · ★★☆*

On the iris data (`load_iris`), write a **hand-written** rule `rule_predict(X)` using only petal length (column 2) and petal width
(column 3) — look at a few class means to choose thresholds. Store its accuracy on all 150 flowers in `acc_rules`
(aim for ≥ 0.9). Then compare with a learned `DecisionTreeClassifier(max_depth=2)` on a train/test split.
What does the learned model do that you did by hand?

In [ ]:
X_iris, y_iris = load_iris(return_X_y=True)
for k in range(3):
    print(k, X_iris[y_iris == k, 2:].mean(0).round(2))

def rule_predict(X):
    # TODO: return an array of class labels 0/1/2
    return None

acc_rules = None

if acc_rules is not None:
    check('rule accuracy >= 0.9', acc_rules >= 0.9 and np.isclose(acc_rules, np.mean(rule_predict(X_iris) == y_iris)), True)
else:
    check('rule accuracy >= 0.9', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Setosa has tiny petals (length < 2.5). Between versicolor and virginica, petal width around 1.7 separates well.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The rule `length < 2.5 → setosa, else width < 1.75 → versicolor, else virginica` reaches 0.96.
A depth-2 tree **learns almost the same rule from data** by greedy impurity minimisation (it splits on petal width $< 0.8$ for setosa, then petal width $< 1.75$). That is the core
idea of ML: the rule is found automatically, and it can be re-learned when data changes. Note that the rule was tuned
on all 150 flowers, so 0.96 is an optimistic estimate; the tree is evaluated on a held-out split.

```python
X_iris, y_iris = load_iris(return_X_y=True)

def rule_predict(X):
    return np.where(X[:, 2] < 2.5, 0, np.where(X[:, 3] < 1.75, 1, 2))

acc_rules = np.mean(rule_predict(X_iris) == y_iris)
Xtr, Xte, ytr, yte = train_test_split(X_iris, y_iris, test_size=0.3, random_state=0, stratify=y_iris)
tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xtr, ytr)
print('rules', acc_rules, '| tree test acc', tree.score(Xte, yte))
print('tree thresholds', tree.tree_.feature[:3], tree.tree_.threshold[:3].round(2))
```

</details>

### Exercise 15 · Performance improves with experience
*💻 Code · ★★☆*

Make Mitchell's definition concrete. On `load_digits`, train `LogisticRegression(max_iter=3000)` on the first
$n\in\{30, 100, 300, 1000\}$ samples of a fixed training split and record test accuracy in a list `acc_by_n`.
Does P improve with E?

In [ ]:
X_d, y_d = load_digits(return_X_y=True)
X_d = X_d / 16.0
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(X_d, y_d, test_size=0.3, random_state=0, stratify=y_d)
sizes = [30, 100, 300, 1000]
acc_by_n = None  # list of 4 test accuracies

if acc_by_n is not None:
    check('accuracy increases with n', all(a < b for a, b in zip(acc_by_n, acc_by_n[1:])), True)
else:
    check('accuracy increases with n', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Loop over `sizes`, fit on `Xd_tr[:n], yd_tr[:n]`, score on the full test set.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Test accuracy climbs from about 0.65 with 30 images to about 0.97 with 1000: the learning curve. With more experience E,
performance P on task T improves, which is exactly Mitchell's definition. The curve flattens: past some point, more data
helps less than a better model ([[Bias-Variance Tradeoff]]).

```python
X_d, y_d = load_digits(return_X_y=True)
X_d = X_d / 16.0
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(X_d, y_d, test_size=0.3, random_state=0, stratify=y_d)
sizes = [30, 100, 300, 1000]
acc_by_n = []
for n in sizes:
    m = LogisticRegression(max_iter=3000).fit(Xd_tr[:n], yd_tr[:n])
    acc_by_n.append(m.score(Xd_te, yd_te))
print(dict(zip(sizes, np.round(acc_by_n, 3))))
```

</details>

### Exercise 16 · Density estimation as loss minimisation
*💻 Code · ★★☆*

Unsupervised tasks fit the same template with $\ell(x) = -\ln p_\theta(x)$. Fit a Gaussian $\mathcal N(\mu, \sigma^2)$ to `x_dens` by
maximum likelihood: implement `gauss_mle(x)` returning `(mu, sigma)` and `avg_nll(x, mu, sigma)`. Compare with `scipy.stats.norm.fit`.

In [ ]:
x_dens = rng.normal(5.0, 2.0, size=200)

def gauss_mle(x):
    return None

def avg_nll(x, mu, sigma):
    return None

theta_dens = gauss_mle(x_dens)

check('MLE (mu, sigma)', theta_dens, stats.norm.fit(x_dens))
if theta_dens is not None:
    check('avg NLL', avg_nll(x_dens, *theta_dens), -stats.norm.logpdf(x_dens, *stats.norm.fit(x_dens)).mean())
else:
    check('avg NLL', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

$\hat\mu = \bar x$, $\hat\sigma^2 = \frac1N\sum(x_i-\bar x)^2$ (divide by $N$, not $N-1$). $-\ln p = \tfrac12\ln(2\pi\sigma^2) + \frac{(x-\mu)^2}{2\sigma^2}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Setting the derivatives of $\frac1N\sum_i[\tfrac12\ln(2\pi\sigma^2) + (x_i-\mu)^2/(2\sigma^2)]$ to zero gives the sample mean and the
**biased** sample variance. At the optimum the average NLL equals $\tfrac12\ln(2\pi\hat\sigma^2) + \tfrac12$.

```python
x_dens = rng.normal(5.0, 2.0, size=200)

def gauss_mle(x):
    mu = x.mean()
    return mu, np.sqrt(np.mean((x - mu) ** 2))

def avg_nll(x, mu, sigma):
    return np.mean(0.5 * np.log(2 * np.pi * sigma ** 2) + (x - mu) ** 2 / (2 * sigma ** 2))

theta_dens = gauss_mle(x_dens)
```

</details>

### Exercise 17 · The hypothesis class matters
*💻 Code · ★★★*

Generate $x\sim U(-3,3)$ (200 points), $y = x^3 - 2x + \varepsilon$, $\varepsilon\sim\mathcal N(0,1)$. Fit least squares with two
representations: (a) $f(x)=w_0+w_1x$ and (b) $f(x) = w_0 + w_1x + w_2x^2 + w_3x^3$. Store the training MSEs in `mse_linear`,
`mse_cubic`. Then explain: the optimiser and the loss are identical — what changed?

In [ ]:
x_h = rng.uniform(-3, 3, 200)
y_h = x_h ** 3 - 2 * x_h + rng.normal(size=200)
mse_linear = None
mse_cubic = None

if mse_linear is not None:
    check('cubic fits much better and near noise level', mse_cubic < 1.3 and mse_linear > 5 * mse_cubic, True)
else:
    check('cubic fits much better and near noise level', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Build design matrices with `np.vander(x_h, 2)` and `np.vander(x_h, 4)` and solve with `np.linalg.lstsq`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Only the **representation** changed. The linear class cannot express a cubic, so even the best member has a large error
(high bias); the cubic class contains the truth and its training MSE drops to the noise level $\sigma^2 = 1$.
Choosing the hypothesis class is the first and often most important modelling decision ([[Feature Engineering]]).

```python
x_h = rng.uniform(-3, 3, 200)
y_h = x_h ** 3 - 2 * x_h + rng.normal(size=200)
def _lsq_mse(Phi, y):
    w = np.linalg.lstsq(Phi, y, rcond=None)[0]
    return np.mean((Phi @ w - y) ** 2)
mse_linear = _lsq_mse(np.vander(x_h, 2), y_h)
mse_cubic = _lsq_mse(np.vander(x_h, 4), y_h)
print(mse_linear, mse_cubic)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [What is Machine Learning](What%20is%20Machine%20Learning.md).*